# 06 — Leave-One-Generator-Out (LOGO)

Evaluates how well AI-image detectors generalize to generators that were not seen during training.

In each LOGO run, one AI generator is completely excluded from the training and validation sets and is used only for the final test evaluation. Real images remain available for training, validation, and testing.

This notebook is parameterized by `UNSEEN_GENERATORS`. A single generator is normally held out per run, while multiple generators can be held out together for specific follow-up experiments.

Both ResNet18 and ViT use the hyperparameters established in notebooks 04–05. No additional hyperparameter tuning is performed here, since the purpose of this notebook is to measure cross-generator generalization rather than optimize in-distribution performance.

## Config & seed

Load the shared experiment configuration and reusable utilities, set the random seed, and select the generator to hold out for this LOGO run.

In [1]:
import json
import sys
from functools import partial
from pathlib import Path
from typing import Callable
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision.transforms import v2


def find_repo_root(start: Path) -> Path:
    """Find the repository root by looking for the experiment config.

    Args:
        start: Directory from which to start searching for the repository root.

    Returns:
        Path to the repository root.

    Raises:
        FileNotFoundError: If the repository root cannot be found.
    """
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "experiments.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT: Path = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.data.config import load_config
from src.data.loaders import build_pretrained_transform, build_vit_transform, create_dataloaders
from src.data.logo import build_logo_manifest, list_ai_generators
from src.evaluation import evaluate_predictions, save_evaluation_outputs
from src.evaluation.classification import EvaluationResult
from src.models import build_resnet18, build_vit, count_trainable_parameters, set_backbone_trainable, unfreeze_vit_blocks
from src.training import evaluate, fit, set_seed, train_one_epoch

config: dict = load_config()
SEED: int = config["seed"]
set_seed(SEED)

device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Generator(s) excluded from training and validation for this LOGO run.
# Normally, one generator is held out at a time. Multiple generators can
# be held out together for deliberate follow-up experiments.
UNSEEN_GENERATORS: list[str] = ["wukong"]
UNSEEN_LABEL: str = "+".join(UNSEEN_GENERATORS)

MANIFEST_PATH: Path = REPO_ROOT / "data" / "dataset.csv"
PROCESSED_ROOT: Path = REPO_ROOT / "data" / "processed"
CHECKPOINT_ROOT: Path = REPO_ROOT / "results" / "checkpoints" / f"logo_{UNSEEN_LABEL}"
METRICS_ROOT: Path = REPO_ROOT / "results" / "metrics" / f"logo_{UNSEEN_LABEL}"
FIGURES_ROOT: Path = REPO_ROOT / "results" / "figures" / f"logo_{UNSEEN_LABEL}"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Unseen generator(s): {UNSEEN_GENERATORS}")

Device: cuda
GPU: NVIDIA GeForce GTX 1660 Ti
Unseen generator(s): ['wukong']


## LOGO split

Build the LOGO manifest by moving all AI images from the held-out generator(s) into the test split and excluding them from training and validation.

The test split contains only real images and AI images from the unseen generator(s). AI images from the remaining generators are used only in training and validation.

The assertions below verify that no held-out generator leaks into training or validation.

In [ ]:
manifest: pd.DataFrame = pd.read_csv(MANIFEST_PATH)
all_generators: list[str] = list_ai_generators(manifest)
print(f"All generators: {all_generators}")
assert all(g in all_generators for g in UNSEEN_GENERATORS), f"{UNSEEN_GENERATORS!r} not all in {all_generators}"

logo_manifest: pd.DataFrame = build_logo_manifest(manifest, UNSEEN_GENERATORS)

print("\nLOGO split sizes:")
print(logo_manifest.groupby(["split", "label"]).size())

train_ai_generators = sorted(logo_manifest.loc[(logo_manifest["split"] == "train") & (logo_manifest["label"] == "ai"), "generator"].unique())
val_ai_generators = sorted(logo_manifest.loc[(logo_manifest["split"] == "val") & (logo_manifest["label"] == "ai"), "generator"].unique())
test_ai_generators = sorted(logo_manifest.loc[(logo_manifest["split"] == "test") & (logo_manifest["label"] == "ai"), "generator"].unique())
print(f"\nTrain AI generators: {train_ai_generators}")
print(f"Val AI generators:   {val_ai_generators}")
print(f"Test AI generators:  {test_ai_generators}")

# Verify that no unseen generator leaks into training or validation.
assert not (set(UNSEEN_GENERATORS) & set(train_ai_generators))
assert not (set(UNSEEN_GENERATORS) & set(val_ai_generators))
assert set(test_ai_generators) == set(UNSEEN_GENERATORS)
print(f"\nConfirmed: {UNSEEN_GENERATORS} are absent from train/val and are the only generators in test.")

All generators: ['adm', 'biggan', 'glide', 'midjourney', 'sdv1_5', 'vqdm', 'wukong']

LOGO split sizes:
split  label
test   ai       1000
       real     1050
train  ai       4200
       real     4900
val    ai        900
       real     1050
dtype: int64

Train AI generators: ['adm', 'biggan', 'glide', 'midjourney', 'sdv1_5', 'vqdm']
Val AI generators:   ['adm', 'biggan', 'glide', 'midjourney', 'sdv1_5', 'vqdm']
Test AI generators:  ['wukong']

Confirmed: ['wukong'] are absent from train/val and are the only generators in test.


## Data loading

Build separate DataLoader sets for ResNet18 and ViT from the same LOGO manifest. Each model uses the preprocessing required by its pretrained weights: ImageNet normalization for ResNet18 and checkpoint-specific normalization for ViT.

The ViT training loader additionally uses the same augmentation configured in notebook 05, while validation and test loaders use deterministic preprocessing only. ResNet18 uses the preprocessing established in notebook 04.

In [ ]:
loader_config: dict = config["dataloader"]
vit_config: dict = config["vit"]

resnet_dataloaders: dict[str, DataLoader] = create_dataloaders(
    manifest=logo_manifest,
    processed_root=PROCESSED_ROOT,
    batch_size=loader_config["batch_size"],
    num_workers=loader_config["num_workers"],
    seed=SEED,
    pin_memory=device.type == "cuda",
    transform_fn=build_pretrained_transform,
)

vit_transform_fn: Callable[[], v2.Compose] = partial(build_vit_transform, vit_config["model_name"])
vit_train_transform_fn: Callable[[], v2.Compose] = partial(build_vit_transform, vit_config["model_name"], augment=True)
vit_dataloaders: dict[str, DataLoader] = create_dataloaders(
    manifest=logo_manifest,
    processed_root=PROCESSED_ROOT,
    batch_size=loader_config["batch_size"],
    num_workers=loader_config["num_workers"],
    seed=SEED,
    pin_memory=device.type == "cuda",
    transform_fn=vit_transform_fn,
    train_transform_fn=vit_train_transform_fn,
)

print("Dataset sizes (identical across the two loader sets):", {split: len(loader.dataset) for split, loader in resnet_dataloaders.items()})

## ResNet18

Use the same two-phase fine-tuning strategy and hyperparameters as in 04_resnet.ipynb: first train the classification head with the backbone frozen, then unfreeze the full backbone and fine-tune the model with head_dropout.

In [ ]:
resnet_config: dict = config["resnet"]
resnet_model: nn.Module = build_resnet18(freeze_backbone=True, head_dropout=resnet_config["head_dropout"]).to(device)
resnet_criterion: nn.BCEWithLogitsLoss = nn.BCEWithLogitsLoss()

# Phase 1: train the classification head with the backbone frozen.
resnet_warmup_optimizer: AdamW = AdamW(
    (p for p in resnet_model.parameters() if p.requires_grad),
    lr=resnet_config["warmup_learning_rate"],
    weight_decay=resnet_config["weight_decay"],
)
for epoch in range(1, resnet_config["warmup_epochs"] + 1):
    train_metrics = train_one_epoch(resnet_model, resnet_dataloaders["train"], resnet_criterion, resnet_warmup_optimizer, device)
    val_metrics = evaluate(resnet_model, resnet_dataloaders["val"], resnet_criterion, device)
    print(
        f"[warmup] epoch {epoch}/{resnet_config['warmup_epochs']} | "
        f"train loss {train_metrics.loss:.4f}, F1 {train_metrics.f1:.4f} | "
        f"val loss {val_metrics.loss:.4f}, F1 {val_metrics.f1:.4f}"
    )

# Phase 2: unfreeze the backbone and fine-tune the full model.
set_backbone_trainable(resnet_model, head_attr="fc", trainable=True)
print(f"Trainable parameters (backbone unfrozen): {count_trainable_parameters(resnet_model):,}")

resnet_finetune_optimizer: AdamW = AdamW(
    resnet_model.parameters(),
    lr=resnet_config["finetune_learning_rate"],
    weight_decay=resnet_config["weight_decay"],
)
resnet_fit_result = fit(
    model=resnet_model,
    train_dataloader=resnet_dataloaders["train"],
    val_dataloader=resnet_dataloaders["val"],
    criterion=resnet_criterion,
    optimizer=resnet_finetune_optimizer,
    device=device,
    epochs=resnet_config["finetune_epochs"],
    patience=resnet_config["early_stopping_patience"],
    min_delta=resnet_config["early_stopping_min_delta"],
    checkpoint_path=CHECKPOINT_ROOT / "resnet18.pt",
)
print(f"Best epoch: {resnet_fit_result.best_epoch}, best val loss: {resnet_fit_result.best_val_loss:.4f}")

resnet_test_result = evaluate_predictions(
    model=resnet_model,
    dataloader=resnet_dataloaders["test"],
    criterion=resnet_criterion,
    device=device,
    threshold=0.5,
)
print(f"\n[{UNSEEN_LABEL}] ResNet18 test metrics:")
display(pd.Series(resnet_test_result.metrics.to_dict(), name="value"))

[warmup] epoch 1/3 | train loss 0.6684, F1 0.5183 | val loss 0.6098, F1 0.5666
[warmup] epoch 2/3 | train loss 0.6211, F1 0.6060 | val loss 0.5983, F1 0.5686
[warmup] epoch 3/3 | train loss 0.6077, F1 0.6239 | val loss 0.5861, F1 0.6542
Trainable parameters (backbone unfrozen): 11,177,025
Epoch 01/20 | train loss 0.5332, F1 0.7124 | val loss 0.4751, F1 0.7561
Epoch 02/20 | train loss 0.4171, F1 0.7997 | val loss 0.4235, F1 0.7853
Epoch 03/20 | train loss 0.3356, F1 0.8450 | val loss 0.4091, F1 0.7938
Epoch 04/20 | train loss 0.2666, F1 0.8846 | val loss 0.3906, F1 0.8084
Epoch 05/20 | train loss 0.2042, F1 0.9186 | val loss 0.3863, F1 0.8191
Epoch 06/20 | train loss 0.1521, F1 0.9472 | val loss 0.3907, F1 0.8121
Epoch 07/20 | train loss 0.1105, F1 0.9652 | val loss 0.3993, F1 0.8214
Epoch 08/20 | train loss 0.0825, F1 0.9792 | val loss 0.4091, F1 0.8151
Epoch 09/20 | train loss 0.0570, F1 0.9878 | val loss 0.4223, F1 0.8226
Epoch 10/20 | train loss 0.0412, F1 0.9933 | val loss 0.4383, 

loss         0.531692
accuracy     0.765366
precision    0.791246
recall       0.705000
f1           0.745637
roc_auc      0.848126
Name: value, dtype: float64

## ViT

Use the same two-phase fine-tuning strategy and hyperparameters as in 05_vit.ipynb: first train the classification head with the backbone frozen, then unfreeze the last unfreeze_last_n_blocks transformer blocks and fine-tune the model.

In [ ]:
vit_model: nn.Module = build_vit(vit_config["model_name"], freeze_backbone=True, drop_path_rate=vit_config["drop_path_rate"]).to(device)
vit_criterion: nn.BCEWithLogitsLoss = nn.BCEWithLogitsLoss()

# Phase 1: train the classification head with the backbone frozen.
vit_warmup_optimizer: AdamW = AdamW(
    (p for p in vit_model.parameters() if p.requires_grad),
    lr=vit_config["warmup_learning_rate"],
    weight_decay=vit_config["weight_decay"],
)
for epoch in range(1, vit_config["warmup_epochs"] + 1):
    train_metrics = train_one_epoch(vit_model, vit_dataloaders["train"], vit_criterion, vit_warmup_optimizer, device)
    val_metrics = evaluate(vit_model, vit_dataloaders["val"], vit_criterion, device)
    print(
        f"[warmup] epoch {epoch}/{vit_config['warmup_epochs']} | "
        f"train loss {train_metrics.loss:.4f}, F1 {train_metrics.f1:.4f} | "
        f"val loss {val_metrics.loss:.4f}, F1 {val_metrics.f1:.4f}"
    )

# Phase 2: unfreeze the last N transformer blocks.
unfreeze_vit_blocks(vit_model, num_blocks=vit_config["unfreeze_last_n_blocks"], head_attr="head")
print(f"Trainable parameters (last {vit_config['unfreeze_last_n_blocks']} blocks unfrozen): "
      f"{count_trainable_parameters(vit_model):,}")

vit_finetune_optimizer: AdamW = AdamW(
    (p for p in vit_model.parameters() if p.requires_grad),
    lr=vit_config["finetune_learning_rate"],
    weight_decay=vit_config["weight_decay"],
)
vit_fit_result = fit(
    model=vit_model,
    train_dataloader=vit_dataloaders["train"],
    val_dataloader=vit_dataloaders["val"],
    criterion=vit_criterion,
    optimizer=vit_finetune_optimizer,
    device=device,
    epochs=vit_config["finetune_epochs"],
    patience=vit_config["early_stopping_patience"],
    min_delta=vit_config["early_stopping_min_delta"],
    checkpoint_path=CHECKPOINT_ROOT / "vit_small.pt",
)
print(f"Best epoch: {vit_fit_result.best_epoch}, best val loss: {vit_fit_result.best_val_loss:.4f}")

vit_test_result = evaluate_predictions(
    model=vit_model,
    dataloader=vit_dataloaders["test"],
    criterion=vit_criterion,
    device=device,
    threshold=0.5,
)
print(f"\n[{UNSEEN_LABEL}] ViT test metrics:")
display(pd.Series(vit_test_result.metrics.to_dict(), name="value"))

[warmup] epoch 1/3 | train loss 0.6118, F1 0.6412 | val loss 0.4968, F1 0.7104
[warmup] epoch 2/3 | train loss 0.4972, F1 0.7347 | val loss 0.4641, F1 0.7370
[warmup] epoch 3/3 | train loss 0.4843, F1 0.7367 | val loss 0.4533, F1 0.7559
Trainable parameters (last 4 blocks unfrozen): 7,099,009
Epoch 01/20 | train loss 0.4341, F1 0.7749 | val loss 0.3396, F1 0.8361
Epoch 02/20 | train loss 0.3223, F1 0.8461 | val loss 0.2999, F1 0.8587
Epoch 03/20 | train loss 0.2634, F1 0.8836 | val loss 0.2911, F1 0.8670
Epoch 04/20 | train loss 0.2106, F1 0.9070 | val loss 0.2788, F1 0.8749
Epoch 05/20 | train loss 0.1855, F1 0.9202 | val loss 0.2996, F1 0.8745
Epoch 06/20 | train loss 0.1616, F1 0.9316 | val loss 0.3167, F1 0.8725
Epoch 07/20 | train loss 0.1353, F1 0.9462 | val loss 0.3129, F1 0.8726
Epoch 08/20 | train loss 0.1241, F1 0.9505 | val loss 0.3268, F1 0.8745
Epoch 09/20 | train loss 0.1161, F1 0.9537 | val loss 0.3186, F1 0.8799
Early stopping after epoch 9; best epoch was 4.
Best epoch

loss         0.597032
accuracy     0.777561
precision    0.868564
recall       0.641000
f1           0.737629
roc_auc      0.878077
Name: value, dtype: float64

## Compare to seen-generator results

Compare each model's performance on unseen generators with its standard test performance, where all generators were represented during training.

The comparison includes accuracy, precision, recall, F1 score, and ROC-AUC. This provides a complete view of how well each model generalizes beyond the generators seen during training.

A small performance drop suggests that the model learned features that generalize across generators. A large drop, particularly in recall, may indicate that the model relies on generator-specific artifacts.

In [ ]:
METRIC_NAMES: list[str] = ["accuracy", "precision", "recall", "f1", "roc_auc"]

logo_results: dict[str, dict[str, float]] = {
    "resnet18": resnet_test_result.metrics.to_dict(),
    "vit_small": vit_test_result.metrics.to_dict(),
}
seen_metric_paths: dict[str, Path] = {
    "resnet18": REPO_ROOT / "results" / "metrics" / "resnet18" / "test_metrics.json",
    "vit_small": REPO_ROOT / "results" / "metrics" / "vit_small" / "test_metrics.json",
}

comparison_rows: list[dict[str, float | str]] = []
for model_name, unseen_metrics in logo_results.items():
    unseen_row: dict[str, float | str] = {"model": model_name, "split": "unseen"}
    unseen_row.update({metric: unseen_metrics[metric] for metric in METRIC_NAMES})
    comparison_rows.append(unseen_row)

    seen_path = seen_metric_paths[model_name]
    if seen_path.is_file():
        seen_metrics = json.loads(seen_path.read_text(encoding="utf-8"))
        seen_row: dict[str, float | str] = {"model": model_name, "split": "seen"}
        seen_row.update({metric: seen_metrics[metric] for metric in METRIC_NAMES})
        comparison_rows.append(seen_row)
    else:
        print(f"{model_name} seen-generator metrics not found at {seen_path.relative_to(REPO_ROOT)}; skipping comparison.")

comparison: pd.DataFrame = pd.DataFrame(comparison_rows).set_index(["model", "split"])
print(f"Unseen generator(s): {UNSEEN_LABEL}")
display(comparison)

## Save results

Save each model's evaluation outputs for the current unseen-generator experiment, including metrics, predictions, and figures.

The results are stored under the corresponding model directory for the current UNSEEN_LABEL. The seen-vs-unseen comparison table is saved separately so that 09_final_comparison.ipynb can aggregate results across different leave-one-generator-out experiments.

In [ ]:
logo_test_results: dict[str, EvaluationResult] = {
    "resnet18": resnet_test_result,
    "vit_small": vit_test_result,
}

for model_name, test_result in logo_test_results.items():
    model_metrics_dir: Path = METRICS_ROOT / model_name
    model_figures_dir: Path = FIGURES_ROOT / model_name
    saved_paths: dict[str, Path] = save_evaluation_outputs(test_result, output_dir=model_metrics_dir, figures_dir=model_figures_dir)
    print(f"Saved {model_name}:")
    for name, path in saved_paths.items():
        print(f"  {name}: {path.relative_to(REPO_ROOT)}")

comparison_path: Path = METRICS_ROOT / "seen_vs_unseen_comparison.csv"
comparison.to_csv(comparison_path)
print(f"\nSaved comparison table: {comparison_path.relative_to(REPO_ROOT)}")

## Findings across generators run so far

This section aggregates the results from all leave-one-generator-out experiments completed so far.

It scans results/metrics/logo_*/seen_vs_unseen_comparison.csv and combines the available results into a single table. This makes the analysis re-runnable and ensures that the summary always reflects the current set of completed generator holdouts without requiring manual updates.

In [ ]:
summary_frames: list[pd.DataFrame] = []
for run_comparison_path in sorted((REPO_ROOT / "results" / "metrics").glob("logo_*/seen_vs_unseen_comparison.csv")):
    run_label: str = run_comparison_path.parent.name.removeprefix("logo_")
    per_run: pd.DataFrame = pd.read_csv(run_comparison_path)
    per_run.insert(0, "unseen_generators", run_label)
    summary_frames.append(per_run)

logo_summary: pd.DataFrame = pd.concat(summary_frames, ignore_index=True).sort_values(
    ["unseen_generators", "model", "split"]
)
display(logo_summary)

summary_path: Path = REPO_ROOT / "results" / "metrics" / "logo_summary.csv"
logo_summary.to_csv(summary_path, index=False)
print(f"Saved: {summary_path.relative_to(REPO_ROOT)}")

### Interpretation

This section summarizes the leave-one-generator-out results collected so far. The table includes the full set of evaluation metrics for each completed run, with the all-generators-seen test results repeated as the baseline for reference.

| Unseen | Model | accuracy | precision | recall | f1 | roc_auc |
|---|---|---|---|---|---|---|
| -- | resnet18 (seen) | 0.819 | 0.799 | 0.852 | 0.825 | 0.906 |
| -- | vit_small (seen) | 0.890 | 0.881 | 0.902 | 0.891 | 0.957 |
| midjourney | resnet18 | 0.636 | 0.750 | 0.380 | 0.504 | 0.739 |
| midjourney | vit_small | 0.690 | 0.837 | 0.452 | 0.587 | 0.833 |
| biggan | resnet18 | 0.813 | 0.807 | 0.810 | 0.808 | 0.887 |
| biggan | vit_small | 0.871 | 0.883 | 0.849 | 0.865 | 0.949 |
| vqdm | resnet18 | 0.602 | 0.718 | 0.305 | 0.428 | 0.683 |
| vqdm | vit_small | 0.608 | 0.769 | 0.280 | 0.411 | 0.732 |
| sdv1_5 (alone) | resnet18 | 0.788 | 0.814 | 0.734 | 0.772 | 0.874 |
| sdv1_5 (alone) | vit_small | 0.819 | 0.894 | 0.714 | 0.794 | 0.921 |
| **wukong (alone)** | resnet18 | 0.765 | 0.791 | 0.705 | 0.746 | 0.848 |
| **wukong (alone)** | vit_small | 0.778 | 0.869 | 0.641 | 0.738 | 0.878 |
| sdv1_5+wukong | resnet18 | 0.541 | 0.857 | 0.361 | 0.508 | 0.727 |
| sdv1_5+wukong | vit_small | 0.546 | 0.901 | 0.346 | 0.500 | 0.788 |
| adm | resnet18 | 0.616 | 0.686 | 0.393 | 0.500 | 0.692 |
| adm | vit_small | 0.601 | 0.770 | 0.261 | 0.390 | 0.709 |
| glide | resnet18 | 0.831 | 0.811 | 0.852 | 0.831 | 0.909 |
| glide | vit_small | 0.906 | 0.907 | 0.899 | 0.903 | 0.963 |

The machine-readable version of these results is maintained automatically in logo_summary and results/metrics/logo_summary.csv.

SDv1.5 and Wukong

The individual holdout results for SDv1.5 and Wukong suggest a symmetric relationship between the two generators. When either generator is held out individually, performance remains in a moderate-to-good range. When both are excluded from training simultaneously, performance drops substantially.

For SDv1.5, recall decreases from 0.714–0.734 when held out individually to 0.346–0.361 when SDv1.5 and Wukong are held out together. Similarly, Wukong reaches a recall of 0.641–0.705 when held out individually, compared with 0.346–0.361 when both generators are excluded.

This pattern is consistent with the hypothesis that the two generators share transferable features related to their architectural lineage. The effect is observed in both directions: SDv1.5 benefits from Wukong being represented during training, and Wukong benefits from SDv1.5 being represented during training.

Overall Generalization Patterns

Across the seven generators, the results suggest three broad generalization patterns:

Strong cross-generator generalization: BigGAN and GLIDE maintain relatively strong performance when held out, suggesting that they contain artifacts that transfer well across generator domains.
Architectural-lineage-dependent generalization: SDv1.5 and Wukong generalize reasonably well when held out individually but deteriorate substantially when both are removed, suggesting that shared architectural characteristics may contribute to transfer.
Weak cross-generator generalization: Midjourney, VQDM, and ADM show substantially lower recall when held out, indicating that the learned representations do not transfer as effectively to these generators.

Precision remains relatively high across the experiments, ranging from 0.686 to 0.907, whereas recall varies much more substantially, from 0.261 to 0.899. This indicates that the main challenge in cross-generator detection is not necessarily producing correct positive predictions, but detecting a sufficiently large proportion of AI-generated images from an unseen generator.

Practical Implications

The LOGO results indicate that cross-generator generalization is different from optimization on a seen-generator test distribution. Hyperparameter and regularization changes that improve performance in notebooks 04–05 cannot necessarily solve poor generalization to a generator that is completely excluded from training.

Potential directions for further investigation include:

More diverse augmentation designed to encourage generator-invariant representations.
Frequency-domain features, building on the frequency-analysis experiments, to investigate the role of low-level artifacts.
Domain-generalization methods that explicitly encourage features to transfer across generator domains.

Overall, the results suggest that cross-generator generalization is influenced by more than one mechanism. Some generators exhibit strong transferable signatures, while others appear to benefit from the presence of related generators during training.